# IFRS S1/S2 Report Pipeline — Phase A (Work Packages) + Phase B (LangGraph Generation)

One notebook, two phases:

**Phase A (deterministic, no LLM)** builds validated, self-contained work packages per report section: enriched requirements mapped to payload keys, pre-computed metrics with provenance, compiled gap directives, the allowed-numbers set, and the entity allowlist.

**Phase B (LangGraph + Azure OpenAI)** runs each section through a gated generation subgraph:

```
plan → draft → deterministic gates (numbers / claimed coverage / meta-commentary)
     → fact judge → coverage judge → style judge → decide
     → (revise → gates → judges ...) up to max_revisions → passed | escalated
```

Model split: **GPT-5.1** drafts and revises; **GPT-5.2** judges. Verification is stricter than generation by design.

Prerequisites: `pip install langgraph langchain-openai python-dotenv pandas`, plus `.env` with `AZURE_OPENAI_ENDPOINT` and `AZURE_OPENAI_API_KEY`.

---
# PART A — Deterministic work-package builder

In [ ]:
# ============================================================
# CELL A1 — SETUP & CONFIG (PHASE A)
# ============================================================

import hashlib
import json
import os
import re
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Optional, TypedDict

import pandas as pd

NOTEBOOK_DIR = Path.cwd()

# ------------------------------------------------------------
# Input locations — point these at your export folders.
# ------------------------------------------------------------
REQUIREMENTS_DIR = NOTEBOOK_DIR / "gen_data" / "requirements"   # *_requirements.json
PAYLOAD_DIR      = NOTEBOOK_DIR / "gen_data" / "payloads"       # payload_BANK01*.json
STYLE_SYSTEM_DIR = NOTEBOOK_DIR / "gen_data" / "style" / "style_system"

PHASE_A_OUTPUT_DIR = NOTEBOOK_DIR / "gen_data" / "generation" / "phase_a"
PHASE_B_OUTPUT_DIR = NOTEBOOK_DIR / "gen_data" / "generation" / "phase_b"

# If everything sits in one flat folder during development, set this and both
# input dirs collapse onto it.
FLAT_INPUT_DIR = None  # e.g. Path(r"C:\Users\...\inputs")
if FLAT_INPUT_DIR is not None:
    REQUIREMENTS_DIR = PAYLOAD_DIR = Path(FLAT_INPUT_DIR)

print("Requirements dir :", REQUIREMENTS_DIR)
print("Payload dir      :", PAYLOAD_DIR)
print("Style system dir :", STYLE_SYSTEM_DIR)
print("Phase A output   :", PHASE_A_OUTPUT_DIR)
print("Phase B output   :", PHASE_B_OUTPUT_DIR)

## A2 — Section registry, tag→payload map, handling rules

`TAG_TO_PAYLOAD_KEYS` maps every evidence tag to the payload keys that can support it; `PARAGRAPH_OVERRIDES` sharpens banking-critical paragraphs (S2.29A, B62/B62A, S1.20/23); `SECTION_DEFAULT_KEYS` + `PARAGRAPH_HANDLING` cover framing paragraphs with no tags (fair presentation, statement of compliance, avoid-duplication) — satisfied by report behaviour or fixed blocks, not data prose.

In [ ]:
SECTION_REGISTRY = {
    "general_requirements": {
        "title": "General Requirements",
        "requirements_file": "general_requirements_requirements.json",
        "payload_file": "payload_BANK01_general_requirements.json",
    },
    "governance": {
        "title": "Governance",
        "requirements_file": "governance_requirements.json",
        "payload_file": "payload_BANK01_governance.json",
    },
    "strategy": {
        "title": "Strategy",
        "requirements_file": "strategy_requirements.json",
        "payload_file": "payload_BANK01_strategy.json",
    },
    "risk_management": {
        "title": "Risk Management",
        "requirements_file": "risk_management_requirements.json",
        "payload_file": "payload_BANK01_risk_management.json",
    },
    "metrics_and_targets": {
        "title": "Metrics and Targets",
        "requirements_file": "metrics_and_targets_requirements.json",
        "payload_file": "payload_BANK01_metrics_targets.json",
    },
}

REQUIRED_REQ_FIELDS = [
    "requirement_id", "standard", "paragraph_id", "report_section",
    "requirement_text", "clause_path", "obligation_type", "mandatory",
    "evidence_tags", "banking_relevance",
]

# ------------------------------------------------------------
# Evidence tag -> payload top-level keys (the requirement->data backbone).
# Keys are intersected with each section's payload slice at build time.
# ------------------------------------------------------------

TAG_TO_PAYLOAD_KEYS = {
    "governance_body": ["governance", "board_minutes", "reporting_kpis.governance_maturity"],
    "management_role": ["governance", "board_minutes"],
    "remuneration": ["governance"],
    "risk_process": ["climate_risk_register", "physical_risk_exposures", "governance",
                     "metadata.risk_rating_methodology"],
    "scenario_analysis": ["climate_scenarios", "resilience_assessment"],
    "business_model_value_chain": ["value_chain_map", "bank", "financial_summary"],
    "strategy_decision_making": ["transition_plan", "climate_opportunities",
                                 "internal_carbon_price", "targets"],
    "financial_effects": ["climate_financial_effects", "financial_summary", "reporting_kpis"],
    "materiality": ["general_requirements_context", "value_chain_map"],
    "connected_information": ["general_requirements_context"],
    "source_guidance": ["general_requirements_context"],
    "metrics": ["reporting_kpis", "scope1", "scope2", "scope3_travel",
                "financed_emissions", "financial_summary"],
    "targets": ["targets", "reporting_kpis.target_summary", "carbon_credits"],
    "ghg_emissions": ["scope1", "scope2", "scope3_travel", "scope3_categories",
                      "ghg_methodology", "scope12_consolidation", "financed_emissions"],
    "scope_1": ["scope1", "ghg_methodology", "metadata.vehicles_correction"],
    "scope_2": ["scope2", "ghg_methodology", "metadata.scope2_rec_reconciliation"],
    "scope_3": ["scope3_travel", "scope3_categories", "financed_emissions"],
    "financed_emissions": ["financed_emissions", "financed_emissions_equity",
                           "financed_emissions_sovereign", "metadata.pcaf_methodology"],
    "commercial_banking": ["financed_emissions", "financed_emissions_equity",
                           "financed_emissions_sovereign", "reporting_kpis"],
    "asset_management": ["financed_emissions_equity"],
    "insurance": [],  # bank archetype has no insurance activities -> conditional N/A
    "carbon_credits": ["carbon_credits", "targets"],
}

# Paragraph-level overrides: sharper mapping for banking-critical paragraphs.
PARAGRAPH_OVERRIDES = {
    ("IFRS S2", "29A"): ["financed_emissions", "financed_emissions_equity",
                         "financed_emissions_sovereign", "metadata.pcaf_methodology",
                         "metadata.data_gaps"],
    ("IFRS S2", "B62"): ["financed_emissions", "financed_emissions_equity",
                         "financed_emissions_sovereign", "metadata.pcaf_methodology"],
    ("IFRS S2", "B62A"): ["financed_emissions_equity", "financed_emissions_sovereign",
                          "metadata.pcaf_methodology"],
    ("IFRS S1", "20"): ["general_requirements_context", "bank"],
    ("IFRS S1", "23"): ["general_requirements_context", "metadata.data_gaps"],
}


# Fallback payload keys for requirements with NO evidence tags (framing/presentation
# paragraphs). Keys are per-section because payload slices differ.
SECTION_DEFAULT_KEYS = {
    "general_requirements": ["general_requirements_context", "metadata.data_gaps", "bank"],
    "governance": ["governance", "board_minutes"],
    "strategy": ["climate_risk_register", "climate_opportunities", "value_chain_map",
                 "climate_scenarios", "climate_financial_effects"],
    "risk_management": ["climate_risk_register", "metadata.risk_rating_methodology", "governance"],
    "metrics_and_targets": ["reporting_kpis", "targets", "ghg_methodology"],
}

# Paragraphs satisfied by PIPELINE BEHAVIOUR rather than generated prose.
# drafting_constraint -> enforced by judges / cross-section pass (e.g. avoid duplication)
# fixed_block        -> deterministic boilerplate block (e.g. statement of compliance)
# conditional_event  -> only applies if the event occurred (period change, error restatement)
PARAGRAPH_HANDLING = {
    ("IFRS S2", "7"): "drafting_constraint",     # avoid duplication with S1
    ("IFRS S2", "26"): "drafting_constraint",    # avoid duplication with S1
    ("IFRS S1", "B37"): "drafting_constraint",   # sensitive-info exemption limits
    ("IFRS S1", "72"): "fixed_block",            # statement of compliance
    ("IFRS S1", "73"): "fixed_block",
    ("IFRS S1", "74"): "fixed_block",
    ("IFRS S1", "66"): "conditional_event",      # change of reporting period end
    ("IFRS S1", "67"): "conditional_event",
    ("IFRS S1", "68"): "conditional_event",
    ("IFRS S1", "69"): "conditional_event",
    ("IFRS S1", "B59"): "conditional_event",     # error restatement impracticable
}

# Tags whose requirements are inherently narrative/process (no numeric payload needed).
NARRATIVE_OK_TAGS = {"connected_information", "source_guidance", "materiality"}

# Archetype-conditional tags: not applicable for a bank without those business lines.
CONDITIONAL_TAGS = {"insurance": "no_insurance_activities", "asset_management": "check_archetype"}

## A3 — Generic helpers

In [ ]:
# ------------------------------------------------------------
# Generic helpers
# ------------------------------------------------------------

def sha256_file(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def load_json(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def get_by_path(obj, dotted: str):
    """Resolve 'metadata.pcaf_methodology' style paths against a dict."""
    cur = obj
    for part in dotted.split("."):
        if isinstance(cur, dict) and part in cur:
            cur = cur[part]
        else:
            return None
    return cur


def fmt_num(value, decimals=1):
    """Canonical display formatting: thousands separators, fixed decimals."""
    if value is None:
        return None
    if decimals == 0:
        return f"{value:,.0f}"
    return f"{value:,.{decimals}f}"

## A4 — Load & validate requirement KBs

In [ ]:
# ------------------------------------------------------------
# Step 1 - Load and validate requirement KBs
# ------------------------------------------------------------

def load_requirements(base_dir: Path):
    kbs, problems = {}, []
    seen_ids = set()
    for key, cfg in SECTION_REGISTRY.items():
        path = base_dir / cfg["requirements_file"]
        if not path.exists():
            problems.append(f"[{key}] requirements file missing: {path}")
            continue
        kb = load_json(path)
        reqs = []
        for std, blk in kb.get("standards", {}).items():
            for r in blk.get("requirements", []):
                missing = [f for f in REQUIRED_REQ_FIELDS if f not in r]
                if missing:
                    problems.append(f"[{key}] {r.get('requirement_id','?')} missing fields {missing}")
                rid = r.get("requirement_id")
                if rid in seen_ids:
                    problems.append(f"[{key}] duplicate requirement_id {rid}")
                seen_ids.add(rid)
                reqs.append(r)
        kbs[key] = {"section_title": kb.get("section_title"), "requirements": reqs,
                    "row_count": len(reqs), "source_file": str(path)}
    return kbs, problems

## A5 — Load & validate payloads (KPI cross-checks against source tables)

In [ ]:
# ------------------------------------------------------------
# Step 2 - Load and validate payload slices + full payload
# ------------------------------------------------------------

def load_payloads(base_dir: Path, full_payload_file="payload_BANK01.json"):
    problems = []
    full = load_json(base_dir / full_payload_file)
    slices = {}
    bank_id = full["metadata"]["bank_id"]
    ryear = full["metadata"]["reporting_year"]
    for key, cfg in SECTION_REGISTRY.items():
        p = base_dir / cfg["payload_file"]
        sl = load_json(p)
        if sl["metadata"]["bank_id"] != bank_id:
            problems.append(f"[{key}] bank_id mismatch")
        if sl["metadata"]["reporting_year"] != ryear:
            problems.append(f"[{key}] reporting_year mismatch")
        slices[key] = sl

    # Cross-consistency: KPI headline figures must equal source tables (2024).
    kpi = full["reporting_kpis"]
    def row_for(table, year):
        return next((r for r in full[table] if r.get("reporting_year") == year), None)

    checks = [
        ("scope1_2024_tco2e", row_for("scope1", ryear), "scope1_total_tco2e"),
        ("scope2_location_2024_tco2e", row_for("scope2", ryear), "scope2_location_tco2e"),
        ("scope2_market_2024_tco2e", row_for("scope2", ryear), "scope2_market_tco2e"),
        ("scope3_travel_2024_tco2e", row_for("scope3_travel", ryear), "scope3_travel_tco2e"),
        ("financed_emissions_2024_tco2e", row_for("financed_emissions", ryear), "financed_em_loans_tco2e"),
    ]
    for kpi_key, row, col in checks:
        if row is None:
            problems.append(f"KPI check: no {ryear} row for {kpi_key}")
            continue
        if abs(kpi[kpi_key] - row[col]) > max(1e-6, abs(row[col]) * 1e-6):
            problems.append(f"KPI mismatch {kpi_key}: kpi={kpi[kpi_key]} table={row[col]}")
    return full, slices, problems

## A6 — Gap directive compiler

Each `metadata.data_gaps` entry becomes an enforceable directive: forbidden claims (hard-gate fodder) + the required disclosure posture — professional IFRS language, no fourth-wall breaks. Gap-following text is scored as **compliant**, never as missing content.

In [ ]:
# ------------------------------------------------------------
# Step 3 - Gap directive compiler
# ------------------------------------------------------------

GAP_FIELD_TO_TAGS = {
    "scope1_fleet_tco2e": ["scope_1", "ghg_emissions", "metrics"],
    "scope3_travel_tco2e": ["scope_3", "ghg_emissions", "metrics"],
    "investments.counterparty_id": ["financed_emissions", "commercial_banking", "asset_management"],
    "total_loans_meur": ["financed_emissions", "metrics", "targets", "commercial_banking"],
}

def compile_gap_directives(full_payload):
    ryear = full_payload["metadata"]["reporting_year"]
    directives = []
    for i, gap in enumerate(full_payload["metadata"]["data_gaps"], start=1):
        field = gap["field"]
        d = {
            "directive_id": f"GAP-{i:02d}",
            "field": field,
            "affected_years": gap["affected_years"],
            "reason": gap["reason"],
            "instruction": gap["instruction"],
            "related_tags": GAP_FIELD_TO_TAGS.get(field, []),
            "forbidden_claims": [],
            "required_disclosure_posture": "",
        }
        if field == "scope1_fleet_tco2e":
            d["forbidden_claims"] = [
                f"Any fleet Scope 1 figure (including zero) for years {gap['affected_years']}",
                "Presenting Scope 1 totals for 2022-2023 as like-for-like comparable with 2024 without noting the fleet coverage change",
            ]
            d["required_disclosure_posture"] = (
                "Report fleet emissions for {} only; state prior-year vehicle activity data is not available; "
                "comparative Scope 1 figures cover stationary combustion (gas) only.".format(ryear))
        elif field == "scope3_travel_tco2e":
            d["forbidden_claims"] = [
                f"Any Scope 3 category 6 figure (including zero) for years {gap['affected_years']}",
                "Any year-on-year trend statement for business travel emissions",
            ]
            d["required_disclosure_posture"] = (
                "Report Scope 3 category 6 for {} only; state comparatives are not available.".format(ryear))
        elif field == "investments.counterparty_id":
            d["forbidden_claims"] = [
                "Issuer-level or counterparty-level claims about investment financed emissions",
                "PCAF data quality better than score 3 for investment attribution",
            ]
            d["required_disclosure_posture"] = (
                "Attribute investment financed emissions at portfolio level; disclose PCAF data quality score 3 "
                "for investment attribution.")
        elif field == "total_loans_meur":
            d["forbidden_claims"] = [
                "Any statement describing growth, decline or trend in the loan book / lending volumes",
                "Attributing carbon intensity changes to changes in lending volume",
            ]
            d["required_disclosure_posture"] = (
                "State explicitly that the financed-emissions intensity trend reflects changes in absolute "
                "emissions only, because the loan-book denominator is held constant in the underlying data; "
                "disclose this as a methodology limitation.")
        directives.append(d)
    return directives

## A7 — Computed metrics engine

All arithmetic happens here; the LLM only re-states `display` strings. Gap-aware: Scope 1 comparatives are gas-only (no fleet figure — including zero — for 2022–23), Scope 3 cat. 6 has no trend metrics, financed-emissions intensity carries the constant-denominator caveat, target progress uses schedule-elapsed proxies flagged as such.

In [ ]:
# ------------------------------------------------------------
# Step 4 - Computed metrics engine (all arithmetic happens HERE, never in the LLM)
# ------------------------------------------------------------

def _metric(mid, name, value, unit, provenance, years=None, decimals=1,
            caveats=None, sections=None):
    return {
        "metric_id": mid,
        "name": name,
        "value": value,
        "display": fmt_num(value, decimals),
        "unit": unit,
        "years": years,
        "provenance": provenance,   # payload paths and/or formula
        "caveats": caveats or [],
        "sections": sections or [],  # which report sections may use it
    }


def compute_metrics(full):
    m = []
    ryear = full["metadata"]["reporting_year"]
    comp_years = full["metadata"]["comparative_years"]
    ALL = ["general_requirements", "governance", "strategy", "risk_management", "metrics_and_targets"]
    MT = ["metrics_and_targets", "general_requirements"]

    def rows(table):
        return sorted(full[table], key=lambda r: r["reporting_year"])

    # --- Scope 1 (gap-aware) ---
    for r in rows("scope1"):
        y = r["reporting_year"]
        cav = []
        if not r["fleet_data_available"]:
            cav = ["Fleet activity data not available for this year; total covers stationary combustion (gas) only. "
                   "Never present a fleet figure (including zero) for this year."]
        m.append(_metric(f"scope1_total_{y}", f"Scope 1 total {y}", r["scope1_total_tco2e"], "tCO2e",
                         {"path": f"scope1[year={y}].scope1_total_tco2e"}, [y], 1, cav, MT + ["strategy"]))
        if r["fleet_data_available"]:
            m.append(_metric(f"scope1_fleet_{y}", f"Scope 1 fleet {y}", r["scope1_fleet_tco2e"], "tCO2e",
                             {"path": f"scope1[year={y}].scope1_fleet_tco2e"}, [y], 1, [], MT))
        m.append(_metric(f"scope1_gas_{y}", f"Scope 1 stationary combustion {y}", r["scope1_gas_tco2e"], "tCO2e",
                         {"path": f"scope1[year={y}].scope1_gas_tco2e"}, [y], 1, [], MT))

    # Gas-only YoY (the only like-for-like Scope 1 comparison permitted)
    s1 = {r["reporting_year"]: r for r in full["scope1"]}
    for y0, y1 in zip(sorted(s1)[:-1], sorted(s1)[1:]):
        delta = (s1[y1]["scope1_gas_tco2e"] - s1[y0]["scope1_gas_tco2e"]) / s1[y0]["scope1_gas_tco2e"] * 100
        m.append(_metric(f"scope1_gas_yoy_{y0}_{y1}", f"Scope 1 gas change {y0}->{y1}", delta, "%",
                         {"formula": f"(gas_{y1}-gas_{y0})/gas_{y0}*100"}, [y0, y1], 1,
                         ["Like-for-like comparison valid for stationary combustion only."], MT))

    # --- Scope 2 ---
    s2 = {r["reporting_year"]: r for r in full["scope2"]}
    for y, r in s2.items():
        m.append(_metric(f"scope2_location_{y}", f"Scope 2 location-based {y}", r["scope2_location_tco2e"], "tCO2e",
                         {"path": f"scope2[year={y}].scope2_location_tco2e"}, [y], 1, [], MT + ["strategy"]))
        m.append(_metric(f"scope2_market_{y}", f"Scope 2 market-based {y}", r["scope2_market_tco2e"], "tCO2e",
                         {"path": f"scope2[year={y}].scope2_market_tco2e"}, [y], 1, [], MT + ["strategy"]))
    for basis in ["location", "market"]:
        ys = sorted(s2)
        for y0, y1 in zip(ys[:-1], ys[1:]):
            a, b = s2[y0][f"scope2_{basis}_tco2e"], s2[y1][f"scope2_{basis}_tco2e"]
            m.append(_metric(f"scope2_{basis}_yoy_{y0}_{y1}", f"Scope 2 {basis}-based change {y0}->{y1}",
                             (b - a) / a * 100, "%", {"formula": f"({basis}_{y1}-{basis}_{y0})/{basis}_{y0}*100"},
                             [y0, y1], 1, [], MT))

    # --- Scope 3 travel (2024 only) ---
    t = full["scope3_travel"][0]
    m.append(_metric(f"scope3_travel_{t['reporting_year']}", "Scope 3 category 6 (business travel)",
                     t["scope3_travel_tco2e"], "tCO2e",
                     {"path": "scope3_travel[year=2024].scope3_travel_tco2e"}, [t["reporting_year"]], 1,
                     ["Comparatives not available (GAP-02). Never state a prior-year travel figure or trend."], MT))

    # --- Total own-operations GHG 2024 (market and location basis) ---
    own_mkt = s1[ryear]["scope1_total_tco2e"] + s2[ryear]["scope2_market_tco2e"] + t["scope3_travel_tco2e"]
    own_loc = s1[ryear]["scope1_total_tco2e"] + s2[ryear]["scope2_location_tco2e"] + t["scope3_travel_tco2e"]
    m.append(_metric(f"own_ops_total_market_{ryear}", "Own operations GHG total (S1 + S2 market + S3 cat.6)",
                     own_mkt, "tCO2e", {"formula": "scope1_total+scope2_market+scope3_travel (2024)"},
                     [ryear], 1, [], MT))
    m.append(_metric(f"own_ops_total_location_{ryear}", "Own operations GHG total (S1 + S2 location + S3 cat.6)",
                     own_loc, "tCO2e", {"formula": "scope1_total+scope2_location+scope3_travel (2024)"},
                     [ryear], 1, [], MT))

    # --- Financed emissions + intensity (denominator caveat from GAP-04) ---
    fe = {r["reporting_year"]: r for r in full["financed_emissions"]}
    denom_caveat = ("Loan-book denominator constant across years in underlying data (GAP-04): intensity trend "
                    "reflects emission changes only; disclose as methodology limitation; never describe loan-book trends.")
    for y, r in fe.items():
        m.append(_metric(f"financed_emissions_loans_{y}", f"Financed emissions - lending {y}",
                         r["financed_em_loans_tco2e"], "tCO2e",
                         {"path": f"financed_emissions[year={y}].financed_em_loans_tco2e"}, [y], 0, [], MT + ["strategy"]))
        m.append(_metric(f"carbon_intensity_lending_{y}", f"Lending carbon intensity {y}",
                         r["carbon_intensity_tco2e_per_meur_lending"], "tCO2e/MEUR",
                         {"path": f"financed_emissions[year={y}].carbon_intensity_tco2e_per_meur_lending"},
                         [y], 1, [denom_caveat], MT + ["strategy"]))
    ys = sorted(fe)
    for y0, y1 in zip(ys[:-1], ys[1:]):
        a, b = fe[y0]["financed_em_loans_tco2e"], fe[y1]["financed_em_loans_tco2e"]
        m.append(_metric(f"financed_emissions_yoy_{y0}_{y1}", f"Financed emissions change {y0}->{y1}",
                         (b - a) / a * 100, "%", {"formula": f"(fe_{y1}-fe_{y0})/fe_{y0}*100"}, [y0, y1], 1,
                         [denom_caveat], MT))

    # --- PCAF weighted data quality scores ---
    for table, label in [("financed_emissions_equity", "equity_investments"),
                         ("financed_emissions_sovereign", "sovereign_bonds")]:
        rs = [r for r in full[table] if r.get("reporting_year", ryear) == ryear] or full[table]
        wsum = sum(r["nominal_amount_meur"] for r in rs)
        dqs = sum(r["pcaf_data_quality_score"] * r["nominal_amount_meur"] for r in rs) / wsum
        m.append(_metric(f"pcaf_dqs_{label}_{ryear}", f"PCAF exposure-weighted DQS - {label}", dqs, "score (1-5)",
                         {"formula": f"sum(dqs*nominal)/sum(nominal) over {table}"}, [ryear], 1,
                         ["Investment attribution at bank level only; PCAF DQS 3 floor applies (GAP-03)."], MT))
        m.append(_metric(f"exposure_{label}_{ryear}", f"Nominal exposure - {label}", wsum, "MEUR",
                         {"formula": f"sum(nominal_amount_meur) over {table}"}, [ryear], 1, [], MT))

    # --- Targets: progress where computable, schedule-elapsed proxies passed through ---
    for ts in full["reporting_kpis"]["target_summary"]:
        tid = ts["target_id"]
        if ts.get("target_progress_pct_2024") is not None:
            m.append(_metric(f"target_progress_{tid}_{ryear}", f"Target {tid} progress", ts["target_progress_pct_2024"],
                             "%", {"path": f"reporting_kpis.target_summary[{tid}].target_progress_pct_2024"},
                             [ryear], 1, [], MT + ["strategy"]))
        if ts.get("schedule_elapsed_pct_2024") is not None:
            m.append(_metric(f"target_schedule_elapsed_{tid}_{ryear}", f"Target {tid} schedule elapsed",
                             ts["schedule_elapsed_pct_2024"], "%",
                             {"path": f"reporting_kpis.target_summary[{tid}].schedule_elapsed_pct_2024"}, [ryear], 1,
                             ["Schedule-elapsed proxy only; never present as emissions progress against the target."],
                             MT + ["strategy"]))
    # Intensity target: measurable progress from baseline (TGT002 baseline 2022 intensity)
    for tg in full["targets"]:
        if tg["metric"] == "tco2e_per_meur_lending" and tg["baseline_year"] in fe:
            base = tg["baseline_value"]
            cur = fe[ryear]["carbon_intensity_tco2e_per_meur_lending"]
            m.append(_metric(f"target_{tg['target_id']}_intensity_change_{ryear}",
                             f"{tg['target_id']} intensity change vs baseline {tg['baseline_year']}",
                             (cur - base) / base * 100, "%",
                             {"formula": f"(intensity_{ryear}-baseline)/baseline*100",
                              "baseline_path": f"targets[{tg['target_id']}].baseline_value"},
                             [tg["baseline_year"], ryear], 1, [denom_caveat], MT + ["strategy"]))

    # --- Governance / board minutes stats ---
    minutes = [b for b in full["board_minutes"] if b["reporting_year"] == ryear]
    for ctype in sorted({b["committee_type"] for b in minutes}):
        ms = [b for b in minutes if b["committee_type"] == ctype]
        clim = [b for b in ms if b["climate_agenda_flag"]]
        dec = [b for b in ms if b["decision_made_flag"]]
        gsec = ["governance", "general_requirements"]
        m.append(_metric(f"meetings_{ctype}_{ryear}", f"{ctype} meetings held {ryear}", len(ms), "count",
                         {"formula": f"count(board_minutes[type={ctype}, year={ryear}])"}, [ryear], 0, [], gsec))
        m.append(_metric(f"meetings_climate_{ctype}_{ryear}", f"{ctype} meetings with climate on agenda {ryear}",
                         len(clim), "count", {"formula": "count(climate_agenda_flag=true)"}, [ryear], 0, [], gsec))
        m.append(_metric(f"meetings_climate_pct_{ctype}_{ryear}", f"{ctype} climate agenda share {ryear}",
                         len(clim) / len(ms) * 100 if ms else None, "%",
                         {"formula": "climate_meetings/total_meetings*100"}, [ryear], 0, [], gsec))
        m.append(_metric(f"decisions_{ctype}_{ryear}", f"{ctype} climate-related decisions {ryear}", len(dec),
                         "count", {"formula": "count(decision_made_flag=true)"}, [ryear], 0, [], gsec))

    # --- Risk register / physical risk aggregates ---
    reg = [r for r in full["climate_risk_register"] if r["reporting_year"] == ryear] or full["climate_risk_register"]
    m.append(_metric(f"risks_registered_{ryear}", "Climate risks in register", len(reg), "count",
                     {"formula": "count(climate_risk_register)"}, [ryear], 0, [],
                     ["risk_management", "strategy", "general_requirements"]))
    for cat in sorted({r["risk_category"] for r in reg}):
        n = len([r for r in reg if r["risk_category"] == cat])
        m.append(_metric(f"risks_{cat}_{ryear}", f"Risks - {cat}", n, "count",
                         {"formula": f"count(risk_category={cat})"}, [ryear], 0, [], ["risk_management"]))
    phys = full.get("physical_risk_exposures", [])
    if phys:
        hi = [p for p in phys if p.get("high_risk_flag")]
        m.append(_metric(f"physical_high_risk_exposure_{ryear}", "High-physical-risk exposure",
                         sum(p["exposure_amount_meur"] for p in hi), "MEUR",
                         {"formula": "sum(exposure_amount_meur where high_risk_flag)"}, [ryear], 1, [],
                         ["risk_management", "strategy"]))
        m.append(_metric(f"physical_exposures_assessed_{ryear}", "Counterparty physical-risk assessments",
                         len(phys), "count", {"formula": "count(physical_risk_exposures)"}, [ryear], 0, [],
                         ["risk_management"]))

    # --- Scope 3 categories screen ---
    cats = [c for c in full["scope3_categories"] if c["reporting_year"] == ryear]
    inc = [c for c in cats if c["included_flag"]]
    m.append(_metric(f"scope3_categories_included_{ryear}", "Scope 3 categories included", len(inc), "count",
                     {"formula": "count(included_flag=true, 2024)"}, [ryear], 0, [], MT))
    m.append(_metric(f"scope3_categories_excluded_{ryear}", "Scope 3 categories excluded", len(cats) - len(inc),
                     "count", {"formula": "count(included_flag=false, 2024)"}, [ryear], 0, [], MT))
    m.append(_metric(f"scope3_included_total_{ryear}", "Scope 3 included categories total",
                     sum(c["emissions_tco2e"] or 0 for c in inc), "tCO2e",
                     {"formula": "sum(emissions_tco2e over included categories, 2024)"}, [ryear], 0, [], MT))

    # --- Carbon credits ---
    cc = [c for c in full["carbon_credits"] if c["reporting_year"] == ryear] or full["carbon_credits"]
    ret = [c for c in cc if c.get("retirement_year")]
    m.append(_metric(f"carbon_credits_tonnes_{ryear}", "Carbon credits volume", sum(c["tonnes_co2e"] for c in cc),
                     "tCO2e", {"formula": "sum(tonnes_co2e)"}, [ryear], 0,
                     ["Distinguish planned vs retired credits per record 'use'/'planned_flag' fields."], MT))
    m.append(_metric(f"carbon_credits_retired_tonnes_{ryear}", "Carbon credits retired",
                     sum(c["tonnes_co2e"] for c in ret), "tCO2e", {"formula": "sum where retirement_year set"},
                     [ryear], 0, [], MT))

    # --- Financial / exposure KPIs passthrough (formatted centrally for consistency) ---
    kpi = full["reporting_kpis"]
    passthrough = [
        ("green_loans_pct_2024", "Green loans share", "%", 1, MT + ["strategy"]),
        ("climate_capex_2024_meur", "Climate-related capex", "MEUR", 1, MT + ["strategy"]),
        ("climate_opex_2024_meur", "Climate-related opex", "MEUR", 1, MT + ["strategy"]),
        ("total_assets_2024_meur", "Total assets", "MEUR", 0, ALL),
        ("total_loans_2024_meur", "Total loans", "MEUR", 0, ALL),
        ("high_carbon_sector_exposure_pct", "High-carbon sector exposure", "%", 1, ["strategy", "risk_management", "metrics_and_targets"]),
        ("fossil_fuel_exposure_pct", "Fossil fuel exposure", "%", 1, ["strategy", "risk_management", "metrics_and_targets"]),
        ("high_carbon_sector_exposure_meur", "High-carbon sector exposure (MEUR)", "MEUR", 1, ["strategy", "risk_management", "metrics_and_targets"]),
        ("fossil_fuel_exposure_meur", "Fossil fuel exposure (MEUR)", "MEUR", 1, ["strategy", "risk_management", "metrics_and_targets"]),
    ]
    for k, name, unit, dec, secs in passthrough:
        if kpi.get(k) is not None:
            m.append(_metric(f"kpi_{k}", name, kpi[k], unit, {"path": f"reporting_kpis.{k}"}, [ryear], dec, [], secs))

    return {x["metric_id"]: x for x in m}

## A8 — Allowed numbers & entity allowlist (inputs to Phase B hard gates)

In [ ]:
# ------------------------------------------------------------
# Step 5 - Allowed numbers + entity allowlist (for downstream hard gates)
# ------------------------------------------------------------

def _num_variants(v):
    out = set()
    if v is None or isinstance(v, bool):
        return out
    try:
        f = float(v)
    except (TypeError, ValueError):
        return out
    for dec in (0, 1, 2):
        out.add(f"{f:,.{dec}f}")
        out.add(f"{f:.{dec}f}")
    if f == int(f):
        out.add(str(int(f)))
    out.add(str(v))
    # thousands-scaled variants (e.g. 35,973,168 tCO2e ~ 36.0 MtCO2e)
    if abs(f) >= 1_000_000:
        out.add(f"{f/1_000_000:,.1f}")
        out.add(f"{f/1_000_000:,.2f}")
    if abs(f) >= 1_000:
        out.add(f"{f/1_000:,.1f}")
    return out


def collect_allowed_numbers(obj, acc=None):
    if acc is None:
        acc = set()
    if isinstance(obj, dict):
        for v in obj.values():
            collect_allowed_numbers(v, acc)
    elif isinstance(obj, list):
        for v in obj:
            collect_allowed_numbers(v, acc)
    elif isinstance(obj, (int, float)) and not isinstance(obj, bool):
        acc |= _num_variants(obj)
    return acc


ENTITY_STRING_KEYS = {
    "bank_name", "assurance_provider", "committee_name", "management_committee_name",
    "scenario_name", "framework", "target_framework", "validation_body", "registry",
    "issuer_name", "standards_basis", "regulatory_regime", "benchmark_reference",
    "aligned_framework", "country", "reporting_entity",
}

def collect_entity_allowlist(obj, acc=None):
    if acc is None:
        acc = set()
    if isinstance(obj, dict):
        for k, v in obj.items():
            if k in ENTITY_STRING_KEYS and isinstance(v, str) and v.strip():
                acc.add(v.strip())
            else:
                collect_entity_allowlist(v, acc)
    elif isinstance(obj, list):
        for v in obj:
            collect_entity_allowlist(v, acc)
    return acc


# ------------------------------------------------------------
# Step 6 - Requirement enrichment (mapping + gap attachment + availability)
# ------------------------------------------------------------

## A9 — Requirement enrichment (mapping, gap attachment, availability status)

In [ ]:
def enrich_requirement(req, payload_slice, gap_directives, bank_archetype, section_key):
    tags = req.get("evidence_tags") or []
    std, pid = req["standard"], req["paragraph_id"]

    mapped = list(PARAGRAPH_OVERRIDES.get((std, pid), []))
    for tg in tags:
        mapped += TAG_TO_PAYLOAD_KEYS.get(tg, [])
    if not mapped and not tags:
        mapped = list(SECTION_DEFAULT_KEYS.get(section_key, []))
    # de-dupe, preserve order
    seen, ordered = set(), []
    for k in mapped:
        if k not in seen:
            seen.add(k)
            ordered.append(k)

    available, unavailable = [], []
    for k in ordered:
        top = k.split(".")[0]
        if top in payload_slice and (get_by_path(payload_slice, k) is not None):
            available.append(k)
        else:
            unavailable.append(k)

    gaps = [g["directive_id"] for g in gap_directives if set(g["related_tags"]) & set(tags)]

    handling = PARAGRAPH_HANDLING.get((std, pid), "disclosure")

    if handling in ("drafting_constraint", "fixed_block", "conditional_event"):
        status = "presentation_rule"
    elif any(t in CONDITIONAL_TAGS for t in tags) and not available:
        status = "not_applicable_archetype" if "insurance" in tags else "conditional_check_archetype"
    elif available:
        status = "data_backed"
    elif set(tags) & NARRATIVE_OK_TAGS or not tags:
        status = "narrative_only"
    else:
        status = "unmapped"  # must be resolved before generation

    return {
        "requirement_id": req["requirement_id"],
        "standard": std,
        "paragraph_id": pid,
        "clause_path": req.get("clause_path"),
        "mandatory": bool(req.get("mandatory")),
        "obligation_type": req.get("obligation_type"),
        "banking_relevance": req.get("banking_relevance"),
        "evidence_tags": tags,
        "requirement_text": req.get("clean_requirement_text") or req["requirement_text"],
        "mapped_payload_keys": available,
        "unavailable_payload_keys": unavailable,
        "gap_directive_ids": gaps,
        "availability_status": status,
        "handling_hint": handling,
    }

## A10 — Work-package assembly & export

In [ ]:
# ------------------------------------------------------------
# Step 7 - Work package assembly + export
# ------------------------------------------------------------

def build_work_packages(base_dir: Path, out_dir: Path, style_dir: Path = None, payload_dir: Path = None):
    payload_dir = payload_dir or base_dir
    out_dir.mkdir(parents=True, exist_ok=True)
    report = {"problems": [], "sections": {}}

    kbs, p1 = load_requirements(base_dir)
    full, slices, p2 = load_payloads(payload_dir)
    report["problems"] += p1 + p2

    gap_directives = compile_gap_directives(full)
    metrics = compute_metrics(full)
    archetype = full["bank"]["archetype"]

    global_numbers = collect_allowed_numbers(full)
    for mt in metrics.values():
        global_numbers |= _num_variants(mt["value"])
    # years and simple context numbers
    for y in [full["metadata"]["reporting_year"]] + full["metadata"]["comparative_years"]:
        global_numbers.add(str(y))
    entities = sorted(collect_entity_allowlist(full))

    manifest = {
        "run_timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "bank_id": full["metadata"]["bank_id"],
        "reporting_year": full["metadata"]["reporting_year"],
        "input_hashes": {},
        "outputs": [],
    }

    for key, cfg in SECTION_REGISTRY.items():
        if key not in kbs:
            continue
        sl = slices[key]
        enriched = [enrich_requirement(r, sl, gap_directives, archetype, key) for r in kbs[key]["requirements"]]

        sec_metrics = {mid: mt for mid, mt in metrics.items() if key in mt["sections"]}
        sec_gap_ids = sorted({g for e in enriched for g in e["gap_directive_ids"]})
        sec_gaps = [g for g in gap_directives if g["directive_id"] in sec_gap_ids] or gap_directives

        style_refs = {}
        if style_dir is not None:
            style_refs = {
                "global_style_guide": str(style_dir / "global_style_guide.json"),
                "section_blueprint": str(style_dir / "authoring" / "section_blueprints" / f"{key}_blueprint.json"),
                "section_style_guide": str(style_dir / "authoring" / "section_style_guides" / f"{key}_style_guide.json"),
                "table_patterns_dir": str(style_dir / "authoring" / "table_patterns"),
                "style_rubric": str(style_dir / "judging" / "style_compliance_rubric.json"),
            }

        wp = {
            "work_package_version": "1.0",
            "section_key": key,
            "section_title": cfg["title"],
            "bank_id": full["metadata"]["bank_id"],
            "reporting_year": full["metadata"]["reporting_year"],
            "comparative_years": full["metadata"]["comparative_years"],
            "requirements": enriched,
            "payload_slice": sl,
            "computed_metrics": sec_metrics,
            "gap_directives": sec_gaps,
            "entity_allowlist": entities,
            "style_artifacts": style_refs,
            "counts": {
                "requirements_total": len(enriched),
                "mandatory": sum(1 for e in enriched if e["mandatory"]),
                "data_backed": sum(1 for e in enriched if e["availability_status"] == "data_backed"),
                "narrative_only": sum(1 for e in enriched if e["availability_status"] == "narrative_only"),
                "presentation_rule": sum(1 for e in enriched if e["availability_status"] == "presentation_rule"),
                "not_applicable_archetype": sum(1 for e in enriched if e["availability_status"] == "not_applicable_archetype"),
                "conditional": sum(1 for e in enriched if e["availability_status"] == "conditional_check_archetype"),
                "unmapped": sum(1 for e in enriched if e["availability_status"] == "unmapped"),
            },
        }
        out_path = out_dir / f"work_package_{key}.json"
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(wp, f, indent=1, ensure_ascii=False)
        manifest["outputs"].append(str(out_path))
        report["sections"][key] = wp["counts"]

        unmapped = [e["requirement_id"] for e in enriched
                    if e["availability_status"] == "unmapped" and e["mandatory"]]
        if unmapped:
            report["problems"].append(f"[{key}] mandatory unmapped requirements: {unmapped}")

    # shared artifacts
    with open(out_dir / "allowed_numbers.json", "w") as f:
        json.dump(sorted(global_numbers), f, indent=0)
    with open(out_dir / "entity_allowlist.json", "w") as f:
        json.dump(entities, f, indent=1, ensure_ascii=False)
    with open(out_dir / "gap_directives.json", "w") as f:
        json.dump(gap_directives, f, indent=1, ensure_ascii=False)
    with open(out_dir / "computed_metrics_all.json", "w") as f:
        json.dump(metrics, f, indent=1, ensure_ascii=False)

    for key, cfg in SECTION_REGISTRY.items():
        for fkey in ["requirements_file", "payload_file"]:
            p = (base_dir if fkey == "requirements_file" else payload_dir) / cfg[fkey]
            if p.exists():
                manifest["input_hashes"][cfg[fkey]] = sha256_file(p)
    manifest["input_hashes"]["payload_BANK01.json"] = sha256_file(payload_dir / "payload_BANK01.json")
    with open(out_dir / "run_manifest.json", "w") as f:
        json.dump(manifest, f, indent=1)

    return report, manifest

## A11 — Run Phase A (hard gates: zero unmapped, zero problems)

In [ ]:
# ============================================================
# CELL A9 — RUN PHASE A + VALIDATION GATES
# ============================================================

phase_a_report, phase_a_manifest = build_work_packages(
    base_dir=REQUIREMENTS_DIR,
    out_dir=PHASE_A_OUTPUT_DIR,
    style_dir=STYLE_SYSTEM_DIR,
    payload_dir=PAYLOAD_DIR,
)

phase_a_summary = pd.DataFrame(phase_a_report["sections"]).T
display(phase_a_summary)

if phase_a_report["problems"]:
    for p in phase_a_report["problems"]:
        print("PROBLEM:", p)
else:
    print("No validation problems.")

# HARD GATES — Phase B is blocked unless these hold.
assert not phase_a_report["problems"], "Phase A validation failed."
assert (phase_a_summary["unmapped"] == 0).all(), "Unmapped mandatory requirements remain."

print(f"\nPhase A complete: {int(phase_a_summary['requirements_total'].sum())} requirements packaged.")

---
# PART B — LangGraph generation engine (Azure OpenAI)

In [ ]:
# ============================================================
# CELL B1 — PHASE B CONFIG (Azure OpenAI GPT-5.1 / GPT-5.2)
# ============================================================
# Credentials come from .env / environment:
#   AZURE_OPENAI_ENDPOINT, AZURE_OPENAI_API_KEY, AZURE_OPENAI_API_VERSION
# Deployment names must match your Azure resource's deployments.

try:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(usecwd=True))
except Exception:
    pass  # .env optional; environment variables may be set directly

from langgraph.graph import StateGraph, END

PHASE_B_CONFIG = {
    "azure_endpoint": os.getenv("AZURE_OPENAI_ENDPOINT", ""),
    "api_key": os.getenv("AZURE_OPENAI_API_KEY", ""),
    "api_version": os.getenv("AZURE_OPENAI_API_VERSION", "2024-12-01-preview"),

    # Model split: GPT-5.1 drafts/revises (cheaper, high volume);
    # GPT-5.2 judges (stronger model verifies — asymmetric verification).
    "drafter_deployment": os.getenv("AZURE_OPENAI_DRAFTER_DEPLOYMENT", "gpt-5.1"),
    "judge_deployment": os.getenv("AZURE_OPENAI_JUDGE_DEPLOYMENT", "gpt-5.2"),

    "max_revisions": 3,
    "style_score_threshold": 85,
    "mock_mode": False,     # True = no API calls; synthetic outputs to test the plumbing
    "request_timeout": 180,
    "max_retries": 3,
}

# Which sections to generate this run (start small while iterating on prompts).
RUN_SECTIONS = [
    "governance",
    "risk_management",
    "general_requirements",
    "strategy",
    "metrics_and_targets",
]

if not PHASE_B_CONFIG["mock_mode"]:
    assert PHASE_B_CONFIG["azure_endpoint"] and PHASE_B_CONFIG["api_key"], \
        "Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_API_KEY (or enable mock_mode)."
print("Drafter deployment:", PHASE_B_CONFIG["drafter_deployment"],
      "| Judge deployment:", PHASE_B_CONFIG["judge_deployment"],
      "| mock_mode:", PHASE_B_CONFIG["mock_mode"])

## B2 — LLM clients and JSON-mode caller

`AzureChatOpenAI` at temperature 0 with JSON response format; `call_json` retries on API/parse failures with exponential backoff. `mock_mode` swaps every LLM call for deterministic synthetic output so the whole graph can be tested without credentials or cost.

In [ ]:
_llm_cache: Dict[str, Any] = {}


def get_llm(role: str):
    """role: 'drafter' | 'judge'. Cached AzureChatOpenAI clients."""
    if PHASE_B_CONFIG["mock_mode"]:
        return None
    if role in _llm_cache:
        return _llm_cache[role]
    from langchain_openai import AzureChatOpenAI
    deployment = (PHASE_B_CONFIG["drafter_deployment"] if role == "drafter"
                  else PHASE_B_CONFIG["judge_deployment"])
    llm = AzureChatOpenAI(
        azure_endpoint=PHASE_B_CONFIG["azure_endpoint"],
        api_key=PHASE_B_CONFIG["api_key"],
        api_version=PHASE_B_CONFIG["api_version"],
        azure_deployment=deployment,
        temperature=0,
        timeout=PHASE_B_CONFIG["request_timeout"],
        model_kwargs={"response_format": {"type": "json_object"}},
    )
    _llm_cache[role] = llm
    return llm


def call_json(role: str, system: str, user: str) -> dict:
    """LLM call that must return a JSON object; retries on parse/API failure."""
    llm = get_llm(role)
    last_err = None
    for attempt in range(PHASE_B_CONFIG["max_retries"]):
        try:
            resp = llm.invoke([{"role": "system", "content": system},
                               {"role": "user", "content": user}])
            text = resp.content if isinstance(resp.content, str) else str(resp.content)
            text = re.sub(r"^```(?:json)?|```$", "", text.strip(), flags=re.M).strip()
            return json.loads(text)
        except Exception as e:  # parse or API error -> retry with backoff
            last_err = e
            time.sleep(2 ** attempt)
    raise RuntimeError(f"call_json({role}) failed after retries: {last_err}")

## B3 — Graph state & deterministic gates

Three mechanical gates that run before any judge:

- **number_gate** — every numeral in a draft must exist in `allowed_numbers` (or be a reporting year / IFRS paragraph reference). A fabricated or re-rounded figure cannot survive this gate.
- **claimed_coverage_gate** — every mandatory requirement ID must be claimed by ≥1 block; blocks may not claim IDs outside the plan.
- **meta_commentary_gate** — the report never mentions data gaps, payloads, pipelines, datasets, prompts. Gap handling must read as professional disclosure language.
- **plan_gate** — every mandatory ID assigned exactly once at planning time; no invented IDs.

In [ ]:
class SectionState(TypedDict, total=False):
    section_key: str
    work_package: dict            # immutable during the run
    plan: dict                    # {"subsections":[{subsection_id,title,requirement_ids}]}
    blocks: List[dict]            # draft blocks (whole section)
    number_report: dict
    claimed_coverage_report: dict
    fact_report: dict
    coverage_report: dict
    style_report: dict
    defects: List[dict]           # aggregated, fed to the reviser
    revision_count: int
    status: str                   # planning|drafting|revising|passed|escalated
    history: List[dict]           # per-iteration gate results (audit trail)


# ============================================================
# Deterministic gates
# ============================================================

_NUM_RE = re.compile(r"(?<![\w./-])(\d{1,3}(?:,\d{3})+(?:\.\d+)?|\d+\.\d+|\d+)(?![\w/-])")
_PARA_REF_RE = re.compile(
    r"(?:paragraph[s]?\s+|paras?\.?\s+|IFRS\s+S[12][\s.,]?\s*(?:paragraph[s]?\s*)?)"
    r"(?:B?\d+[A-Z]?(?:\([a-z0-9ivx]+\))*)(?:[-\u2013]B?\d+[A-Z]?)?",
    re.IGNORECASE)


def number_gate(blocks: List[dict], allowed_numbers: set, reporting_years: set) -> dict:
    """Every numeral in prose must be an allowed value, a reporting year,
    or part of an IFRS paragraph reference."""
    violations = []
    for b in blocks:
        text = b.get("text", "")
        # mask paragraph references so their numbers are not flagged
        masked = _PARA_REF_RE.sub(" <PARAREF> ", text)
        for mnum in _NUM_RE.finditer(masked):
            tok = mnum.group(1)
            if tok in allowed_numbers or tok in reporting_years:
                continue
            plain = tok.replace(",", "")
            if plain in allowed_numbers:
                continue
            violations.append({"block_id": b.get("block_id"), "kind": "number_not_allowed",
                               "detail": f"'{tok}' not in the allowed value set",
                               "gate": "number_gate"})
    return {"passed": not violations, "violations": violations}


def claimed_coverage_gate(blocks: List[dict], plan: dict, mandatory_ids: set) -> dict:
    """All mandatory requirement IDs must be claimed by at least one block,
    and blocks may only claim IDs that exist in the plan."""
    claimed = set()
    unknown = []
    planned_ids = {rid for s in plan.get("subsections", []) for rid in s.get("requirement_ids", [])}
    for b in blocks:
        for rid in b.get("requirement_ids", []):
            claimed.add(rid)
            if rid not in planned_ids:
                unknown.append({"block_id": b.get("block_id"), "kind": "unknown_requirement_id",
                                "detail": rid, "gate": "claimed_coverage_gate"})
    missing = sorted(mandatory_ids - claimed)
    violations = unknown + [{"block_id": None, "kind": "mandatory_id_not_claimed",
                             "detail": rid, "gate": "claimed_coverage_gate"} for rid in missing]
    return {"passed": not violations, "violations": violations,
            "claimed": sorted(claimed), "missing": missing}


def plan_gate(plan: dict, mandatory_ids: set, all_ids: set) -> dict:
    """Every mandatory ID assigned exactly once; no invented IDs."""
    seen, dups, invented = {}, [], []
    for s in plan.get("subsections", []):
        for rid in s.get("requirement_ids", []):
            if rid not in all_ids:
                invented.append(rid)
            seen[rid] = seen.get(rid, 0) + 1
            if seen[rid] == 2:
                dups.append(rid)
    missing = sorted(mandatory_ids - set(seen))
    ok = not (missing or dups or invented)
    return {"passed": ok, "missing": missing, "duplicates": dups, "invented": invented}


META_COMMENTARY_PATTERNS = [
    r"\bdata gap[s]?\b", r"\bpayload\b", r"\bwork package\b", r"\bpipeline\b",
    r"\bLLM\b", r"\bprompt\b", r"\bsynthetic\b", r"\bplaceholder\b",
    r"\bthe dataset\b", r"\bour dataset\b",
]

def meta_commentary_gate(blocks: List[dict]) -> dict:
    """The report must never break the fourth wall."""
    violations = []
    for b in blocks:
        for pat in META_COMMENTARY_PATTERNS:
            if re.search(pat, b.get("text", ""), re.IGNORECASE):
                violations.append({"block_id": b.get("block_id"), "kind": "meta_commentary",
                                   "detail": f"matched '{pat}'", "gate": "meta_commentary_gate"})
    return {"passed": not violations, "violations": violations}

## B4 — Prompts & context builders

The drafter works subsection-by-subsection with a **closed world**: only the payload keys mapped to its requirements, the computed metrics, the binding gap directives, the entity allowlist, and the style rules. Every factual claim must carry citations (payload paths / metric IDs) — that's what makes the fact judge's job checkable. Judges get the full section draft plus the evidence to verify against.

In [ ]:
def _req_brief(reqs: List[dict], with_text=True, max_chars=280) -> str:
    lines = []
    for r in reqs:
        t = (r["requirement_text"][:max_chars] + "...") if with_text and len(r["requirement_text"]) > max_chars \
            else (r["requirement_text"] if with_text else "")
        lines.append(f"- {r['requirement_id']} [{r['standard']} {r['paragraph_id']} {r['clause_path'] or ''}]"
                     f" (status={r['availability_status']}, handling={r['handling_hint']},"
                     f" keys={r['mapped_payload_keys']}, gaps={r['gap_directive_ids']}) {t}")
    return "\n".join(lines)


PLANNER_SYSTEM = """You are a senior IFRS S1/S2 sustainability reporting architect planning ONE section of a bank's report.
Group the requirements into a coherent subsection outline a reader would expect in a professional disclosure document.
Rules:
- Assign EVERY requirement_id you are given to EXACTLY ONE subsection. Do not invent, drop, or duplicate IDs.
- 3 to 9 subsections. Order them for narrative flow (context -> processes -> quantitative disclosures -> limitations/assurance).
- Requirements with handling 'fixed_block' or 'drafting_constraint' or 'conditional_event' go into a final subsection titled "Basis of preparation and compliance statements".
- Return JSON only: {"subsections":[{"subsection_id":"SUB-01","title":"...","requirement_ids":["..."]}]}"""


def build_planner_user(wp: dict) -> str:
    blueprint = ""
    bp_path = wp.get("style_artifacts", {}).get("section_blueprint")
    if bp_path and Path(bp_path).exists():
        blueprint = "SECTION BLUEPRINT (structure inspiration only):\n" + Path(bp_path).read_text(encoding="utf-8")[:4000]
    return f"""SECTION: {wp['section_title']} ({wp['section_key']}) | Bank: {wp['bank_id']} | Year: {wp['reporting_year']}

REQUIREMENTS TO ASSIGN ({len(wp['requirements'])}):
{_req_brief(wp['requirements'], with_text=True)}

{blueprint}"""


DRAFTER_SYSTEM = """You write one subsection of an IFRS S1/S2 sustainability report for a bank. You are given the requirements this subsection must satisfy, the exact payload data mapped to them, pre-computed metrics, gap directives, and style rules.

NON-NEGOTIABLE RULES:
1. CLOSED WORLD. Use ONLY facts present in the provided payload extract and computed metrics. No outside knowledge about the bank. Generic descriptions of IFRS requirements themselves are allowed.
2. NUMBERS. Use numeric values EXACTLY as they appear in computed_metrics 'display' strings or the payload. Never compute, round differently, aggregate, or estimate a number yourself.
3. CITATIONS. Every factual claim about the bank must carry citations: payload paths (e.g. "governance.esg_committee_meetings_per_year") and/or metric_ids. A sentence without support must be either removed or rewritten as a generic standards description with citations [].
4. GAP DIRECTIVES. Follow each directive's required_disclosure_posture using professional disclosure language. NEVER produce any forbidden claim. NEVER mention data gaps, datasets, payloads, pipelines, or any meta-commentary — the report speaks only as a formal disclosure document (e.g. "Comparative information is not presented because vehicle activity data became available in 2024.").
5. ENTITIES. Name only entities in the entity allowlist. No invented committees, vendors, frameworks, or people. Never name individuals.
6. STYLE. Follow the style rules provided: formal, precise, neutral, decision-useful; no promotional language, no guarantees about future outcomes.
7. Each requirement assigned to this subsection must be substantively addressed by at least one block that lists it in requirement_ids. 'fixed_block' requirements -> write the formal compliance statement. 'conditional_event' requirements -> address only if the payload shows the event occurred; otherwise cover them with the basis-of-preparation statement block (still listing their IDs). 'not_applicable_archetype' -> one sentence stating the disclosure is not applicable to the bank's activities.

Return JSON only:
{"blocks":[{"block_id":"B-<subsection>-01","type":"heading|paragraph|table_note","text":"...","requirement_ids":["..."],"citations":["path.or.metric_id", "..."]}]}"""


def _payload_extract(wp: dict, keys: List[str], budget_chars=14000) -> str:
    """Serialize only the payload keys needed by this subsection, within budget."""
    out, used = {}, 0
    for k in keys:
        top = k.split(".")[0]
        if top in out:
            continue
        val = wp["payload_slice"].get(top)
        if val is None:
            continue
        s = json.dumps(val, ensure_ascii=False)
        if used + len(s) > budget_chars and out:
            out[top] = f"<truncated - {len(s)} chars; rely on computed_metrics for figures>"
            continue
        out[top] = val
        used += len(s)
    return json.dumps(out, indent=1, ensure_ascii=False)


def build_drafter_user(wp: dict, subsection: dict, style_rules: str) -> str:
    reqs = [r for r in wp["requirements"] if r["requirement_id"] in set(subsection["requirement_ids"])]
    keys = sorted({k for r in reqs for k in r["mapped_payload_keys"]})
    gap_ids = sorted({g for r in reqs for g in r["gap_directive_ids"]})
    gaps = [g for g in wp["gap_directives"] if g["directive_id"] in gap_ids] or wp["gap_directives"]
    metrics = {mid: {kk: m[kk] for kk in ("name", "display", "unit", "years", "caveats")}
               for mid, m in wp["computed_metrics"].items()}
    return f"""SUBSECTION: {subsection['title']} ({subsection['subsection_id']}) of section '{wp['section_title']}'
BANK: {wp['bank_id']} | REPORTING YEAR: {wp['reporting_year']} | COMPARATIVE YEARS: {wp['comparative_years']}

REQUIREMENTS TO SATISFY:
{_req_brief(reqs)}

GAP DIRECTIVES (binding):
{json.dumps(gaps, indent=1, ensure_ascii=False)}

COMPUTED METRICS (the ONLY derived figures you may state; cite by metric_id):
{json.dumps(metrics, indent=1, ensure_ascii=False)}

PAYLOAD EXTRACT (cite by path):
{_payload_extract(wp, keys)}

ENTITY ALLOWLIST:
{json.dumps(wp['entity_allowlist'], ensure_ascii=False)}

STYLE RULES:
{style_rules}"""


FACT_JUDGE_SYSTEM = """You are an independent disclosure verification auditor. For the draft blocks provided, check EVERY factual claim about the bank:
1. supported: the cited payload path / metric supports the claim as written (numbers exact, qualifiers correct).
2. uncited: factual claims with empty citations that are not generic descriptions of IFRS requirements.
3. forbidden: any statement matching a gap directive's forbidden_claims.
4. entity: any named entity not in the allowlist.
5. posture: where a gap directive applies to the block's requirements, the required_disclosure_posture is honoured.
Be strict: unverifiable = violation. Return JSON only:
{"verdict":"pass|fail","violations":[{"block_id":"...","kind":"unsupported|uncited|forbidden|entity|posture|wrong_number","detail":"..."}]}"""


COVERAGE_JUDGE_SYSTEM = """You are an IFRS S1/S2 compliance reviewer. For each requirement, judge whether the blocks claiming it SUBSTANTIVELY address it (per its text and clause path), considering handling hints:
- 'disclosure': needs substantive content.
- 'fixed_block': a formal compliance/basis statement suffices.
- 'drafting_constraint'/'conditional_event': a basis-of-preparation statement suffices.
- Requirements whose satisfaction is limited by a gap directive count as SATISFIED if the required posture is followed. Do not penalize absent data covered by a directive.
Return JSON only:
{"verdict":"pass|fail","weak":[{"requirement_id":"...","detail":"..."}],"missing":[{"requirement_id":"...","detail":"..."}]}"""


STYLE_JUDGE_SYSTEM = """You are a sustainability-report style auditor. Score the draft against the style rules and rubric provided (0-100). Any violation of a 'do_not_do' rule is an automatic fail regardless of score. Never penalize compliant handling of unavailable data (professional statements that information is not presented). Return JSON only:
{"score": <int>, "auto_fail": <bool>, "verdict":"pass|fail", "violations":[{"block_id":"...","detail":"..."}]}"""


REVISER_SYSTEM = """You are revising blocks of an IFRS S1/S2 report subsection to fix the exact defects listed, under the same non-negotiable rules as the original drafter (closed world, exact numbers, citations, gap directives, no meta-commentary, entity allowlist, style rules). Change ONLY what the defects require; keep block_ids stable; keep every requirement_id covered. Return JSON only, same schema:
{"blocks":[{"block_id":"...","type":"...","text":"...","requirement_ids":[...],"citations":[...]}]}"""

## B5 — Style rules loader & mock outputs

In [ ]:
def load_style_rules(wp: dict, phase_a_dir: Optional[Path] = None) -> str:
    """Global style guide (+ section style guide if present)."""
    texts = []
    gpath = wp.get("style_artifacts", {}).get("global_style_guide")
    candidates = [Path(gpath)] if gpath else []
    if phase_a_dir:
        candidates.append(Path(phase_a_dir).parent.parent / "global_style_guide.json")
    for c in candidates:
        if c and c.exists():
            texts.append(c.read_text(encoding="utf-8"))
            break
    spath = wp.get("style_artifacts", {}).get("section_style_guide")
    if spath and Path(spath).exists():
        texts.append(Path(spath).read_text(encoding="utf-8")[:4000])
    return "\n\n".join(texts) if texts else "Formal, precise, neutral, audit-ready disclosure voice."


# ============================================================
# Mock outputs (plumbing tests without API calls)
# ============================================================

def _mock_plan(wp):
    reqs = [r["requirement_id"] for r in wp["requirements"]]
    special = [r["requirement_id"] for r in wp["requirements"]
               if r["handling_hint"] in ("fixed_block", "drafting_constraint", "conditional_event")]
    core = [r for r in reqs if r not in special]
    half = max(1, len(core) // 2)
    subs = [{"subsection_id": "SUB-01", "title": "Approach and processes", "requirement_ids": core[:half]},
            {"subsection_id": "SUB-02", "title": "Quantitative disclosures", "requirement_ids": core[half:]}]
    if special:
        subs.append({"subsection_id": "SUB-03", "title": "Basis of preparation and compliance statements",
                     "requirement_ids": special})
    return {"subsections": [s for s in subs if s["requirement_ids"]]}


def _mock_blocks(wp, subsection):
    mid, mt = next(iter(wp["computed_metrics"].items())) if wp["computed_metrics"] else (None, None)
    num = f" The reported figure is {mt['display']} {mt['unit']}." if mt else ""
    cites = [f"metric:{mid}"] if mid else []
    return {"blocks": [
        {"block_id": f"B-{subsection['subsection_id']}-01", "type": "heading",
         "text": subsection["title"], "requirement_ids": [], "citations": []},
        {"block_id": f"B-{subsection['subsection_id']}-02", "type": "paragraph",
         "text": ("The Group discloses the information required for this area in accordance with "
                  "IFRS S1 and IFRS S2." + num),
         "requirement_ids": subsection["requirement_ids"], "citations": cites},
    ]}

## B6 — Graph nodes

`plan` (validated by plan_gate, one retry) → `draft` (per subsection) → `deterministic_gates` → `fact_judge` → `coverage_judge` → `style_judge` → `decide` (pass / revise / escalate at `max_revisions`) → `revise` loops back to the gates. Escalation never silently passes — open defects are persisted for human review.

In [ ]:
def node_plan(state: SectionState) -> dict:
    wp = state["work_package"]
    all_ids = {r["requirement_id"] for r in wp["requirements"]}
    mandatory = {r["requirement_id"] for r in wp["requirements"] if r["mandatory"]}
    for attempt in range(2):
        plan = (_mock_plan(wp) if PHASE_B_CONFIG["mock_mode"]
                else call_json("drafter", PLANNER_SYSTEM, build_planner_user(wp)))
        gate = plan_gate(plan, mandatory, all_ids)
        if gate["passed"]:
            return {"plan": plan, "status": "drafting"}
    raise RuntimeError(f"[{state['section_key']}] plan gate failed twice: {gate}")


def node_draft(state: SectionState) -> dict:
    wp = state["work_package"]
    style_rules = state.get("_style_rules") or load_style_rules(wp)
    blocks = []
    for sub in state["plan"]["subsections"]:
        out = (_mock_blocks(wp, sub) if PHASE_B_CONFIG["mock_mode"]
               else call_json("drafter", DRAFTER_SYSTEM, build_drafter_user(wp, sub, style_rules)))
        blocks.extend(out["blocks"])
    return {"blocks": blocks, "revision_count": state.get("revision_count", 0)}


def node_deterministic_gates(state: SectionState) -> dict:
    wp = state["work_package"]
    allowed = set(wp.get("_allowed_numbers", []))
    years = {str(wp["reporting_year"])} | {str(y) for y in wp["comparative_years"]}
    mandatory = {r["requirement_id"] for r in wp["requirements"] if r["mandatory"]}
    ngate = number_gate(state["blocks"], allowed, years)
    cgate = claimed_coverage_gate(state["blocks"], state["plan"], mandatory)
    mgate = meta_commentary_gate(state["blocks"])
    defects = ngate["violations"] + cgate["violations"] + mgate["violations"]
    return {"number_report": ngate, "claimed_coverage_report": cgate, "defects": defects}


def _blocks_for_judges(state) -> str:
    return json.dumps({"blocks": state["blocks"]}, indent=1, ensure_ascii=False)


def node_fact_judge(state: SectionState) -> dict:
    if PHASE_B_CONFIG["mock_mode"]:
        return {"fact_report": {"verdict": "pass", "violations": []}}
    wp = state["work_package"]
    keys = sorted({k for r in wp["requirements"] for k in r["mapped_payload_keys"]})
    user = f"""DRAFT BLOCKS:
{_blocks_for_judges(state)}

GAP DIRECTIVES:
{json.dumps(wp['gap_directives'], indent=1, ensure_ascii=False)}

COMPUTED METRICS:
{json.dumps({m: {k: v[k] for k in ('display','unit','years','caveats')} for m, v in wp['computed_metrics'].items()}, indent=1, ensure_ascii=False)}

PAYLOAD:
{_payload_extract(wp, keys, budget_chars=30000)}

ENTITY ALLOWLIST:
{json.dumps(wp['entity_allowlist'], ensure_ascii=False)}"""
    return {"fact_report": call_json("judge", FACT_JUDGE_SYSTEM, user)}


def node_coverage_judge(state: SectionState) -> dict:
    if PHASE_B_CONFIG["mock_mode"]:
        return {"coverage_report": {"verdict": "pass", "weak": [], "missing": []}}
    wp = state["work_package"]
    user = f"""REQUIREMENTS:
{_req_brief(wp['requirements'], max_chars=400)}

GAP DIRECTIVES:
{json.dumps(wp['gap_directives'], indent=1, ensure_ascii=False)}

DRAFT BLOCKS:
{_blocks_for_judges(state)}"""
    return {"coverage_report": call_json("judge", COVERAGE_JUDGE_SYSTEM, user)}


def node_style_judge(state: SectionState) -> dict:
    if PHASE_B_CONFIG["mock_mode"]:
        return {"style_report": {"score": 95, "auto_fail": False, "verdict": "pass", "violations": []}}
    wp = state["work_package"]
    style_rules = state.get("_style_rules") or load_style_rules(wp)
    rubric = ""
    rpath = wp.get("style_artifacts", {}).get("style_rubric")
    if rpath and Path(rpath).exists():
        rubric = Path(rpath).read_text(encoding="utf-8")[:6000]
    user = f"""STYLE RULES:
{style_rules}

RUBRIC:
{rubric}

DRAFT BLOCKS:
{_blocks_for_judges(state)}"""
    return {"style_report": call_json("judge", STYLE_JUDGE_SYSTEM, user)}


def node_decide(state: SectionState) -> dict:
    defects = list(state.get("defects", []))
    fr = state.get("fact_report", {})
    if fr.get("verdict") == "fail":
        defects += [dict(v, gate="fact_judge") for v in fr.get("violations", [])]
    cr = state.get("coverage_report", {})
    if cr.get("verdict") == "fail":
        defects += [{"block_id": None, "kind": "weak_coverage", "detail": json.dumps(w), "gate": "coverage_judge"}
                    for w in cr.get("weak", []) + cr.get("missing", [])]
    sr = state.get("style_report", {})
    if sr.get("auto_fail") or sr.get("score", 100) < PHASE_B_CONFIG["style_score_threshold"]:
        defects += [dict(v, gate="style_judge") for v in sr.get("violations", [])]

    hist = state.get("history", []) + [{
        "revision": state.get("revision_count", 0),
        "n_defects": len(defects),
        "style_score": sr.get("score"),
        "gates": sorted({d["gate"] for d in defects}),
    }]
    if not defects:
        return {"status": "passed", "defects": [], "history": hist}
    if state.get("revision_count", 0) >= PHASE_B_CONFIG["max_revisions"]:
        return {"status": "escalated", "defects": defects, "history": hist}
    return {"status": "revising", "defects": defects, "history": hist}


def node_revise(state: SectionState) -> dict:
    wp = state["work_package"]
    if PHASE_B_CONFIG["mock_mode"]:
        return {"blocks": state["blocks"], "revision_count": state.get("revision_count", 0) + 1}
    style_rules = state.get("_style_rules") or load_style_rules(wp)
    keys = sorted({k for r in wp["requirements"] for k in r["mapped_payload_keys"]})
    user = f"""DEFECTS TO FIX:
{json.dumps(state['defects'], indent=1, ensure_ascii=False)}

CURRENT BLOCKS:
{_blocks_for_judges(state)}

GAP DIRECTIVES:
{json.dumps(wp['gap_directives'], indent=1, ensure_ascii=False)}

COMPUTED METRICS:
{json.dumps({m: {k: v[k] for k in ('display','unit','years','caveats')} for m, v in wp['computed_metrics'].items()}, indent=1, ensure_ascii=False)}

PAYLOAD EXTRACT:
{_payload_extract(wp, keys, budget_chars=20000)}

ENTITY ALLOWLIST:
{json.dumps(wp['entity_allowlist'], ensure_ascii=False)}

STYLE RULES:
{style_rules}"""
    out = call_json("drafter", REVISER_SYSTEM, user)
    return {"blocks": out["blocks"], "revision_count": state.get("revision_count", 0) + 1}


def route_after_decide(state: SectionState) -> str:
    return {"passed": END, "escalated": END}.get(state["status"], "revise")

## B7 — Graph assembly & section driver

`run_section` loads a Phase A work package, injects the allowed-numbers set, invokes the graph, and persists `draft_<section>.json` (plan + blocks + result) and `gate_reports_<section>.json` (full audit trail of every gate/judge).

In [ ]:
def build_section_graph():
    g = StateGraph(SectionState)
    g.add_node("plan", node_plan)
    g.add_node("draft", node_draft)
    g.add_node("deterministic_gates", node_deterministic_gates)
    g.add_node("fact_judge", node_fact_judge)
    g.add_node("coverage_judge", node_coverage_judge)
    g.add_node("style_judge", node_style_judge)
    g.add_node("decide", node_decide)
    g.add_node("revise", node_revise)

    g.set_entry_point("plan")
    g.add_edge("plan", "draft")
    g.add_edge("draft", "deterministic_gates")
    g.add_edge("deterministic_gates", "fact_judge")
    g.add_edge("fact_judge", "coverage_judge")
    g.add_edge("coverage_judge", "style_judge")
    g.add_edge("style_judge", "decide")
    g.add_conditional_edges("decide", route_after_decide, {"revise": "revise", END: END})
    g.add_edge("revise", "deterministic_gates")
    return g.compile()


def run_section(graph, work_package_path: Path, allowed_numbers_path: Path,
                out_dir: Path, style_rules_override: Optional[str] = None) -> dict:
    wp = json.loads(Path(work_package_path).read_text(encoding="utf-8"))
    wp["_allowed_numbers"] = json.loads(Path(allowed_numbers_path).read_text(encoding="utf-8"))
    init: SectionState = {
        "section_key": wp["section_key"],
        "work_package": wp,
        "revision_count": 0,
        "status": "planning",
        "history": [],
    }
    if style_rules_override:
        init["_style_rules"] = style_rules_override
    final = graph.invoke(init, config={"recursion_limit": 60})

    out_dir.mkdir(parents=True, exist_ok=True)
    result = {
        "section_key": wp["section_key"],
        "status": final["status"],
        "revision_count": final.get("revision_count", 0),
        "style_score": final.get("style_report", {}).get("score"),
        "n_blocks": len(final.get("blocks", [])),
        "open_defects": final.get("defects", []),
        "history": final.get("history", []),
    }
    (out_dir / f"draft_{wp['section_key']}.json").write_text(
        json.dumps({"section_key": wp["section_key"], "plan": final.get("plan"),
                    "blocks": final.get("blocks", []), "result": result},
                   indent=1, ensure_ascii=False), encoding="utf-8")
    (out_dir / f"gate_reports_{wp['section_key']}.json").write_text(
        json.dumps({k: final.get(k) for k in
                    ("number_report", "claimed_coverage_report", "fact_report",
                     "coverage_report", "style_report")}, indent=1, ensure_ascii=False),
        encoding="utf-8")
    return result

## B8 — Run generation + scorecard

Start with one or two sections in `RUN_SECTIONS` while iterating on prompts; Metrics & Targets (151 requirements) is the most expensive. Any `escalated` section ships its defect list in the gate reports — fix inputs/prompts and re-run just that section.

In [ ]:
# ============================================================
# CELL B8 — RUN PHASE B OVER SELECTED SECTIONS + SCORECARD
# ============================================================

section_graph = build_section_graph()

results = []
for sec in RUN_SECTIONS:
    wp_path = PHASE_A_OUTPUT_DIR / f"work_package_{sec}.json"
    assert wp_path.exists(), f"Run Phase A first — missing {wp_path}"
    print(f"--- generating section: {sec} ---")
    r = run_section(
        section_graph,
        work_package_path=wp_path,
        allowed_numbers_path=PHASE_A_OUTPUT_DIR / "allowed_numbers.json",
        out_dir=PHASE_B_OUTPUT_DIR,
    )
    results.append(r)
    print(f"    status={r['status']} blocks={r['n_blocks']} revisions={r['revision_count']}"
          f" style={r['style_score']} open_defects={len(r['open_defects'])}")

scorecard = pd.DataFrame([{k: r[k] for k in
    ("section_key", "status", "n_blocks", "revision_count", "style_score")} for r in results])
display(scorecard)

(PHASE_B_OUTPUT_DIR / "scorecard.json").write_text(
    json.dumps(results, indent=1, ensure_ascii=False), encoding="utf-8")

escalated = [r["section_key"] for r in results if r["status"] != "passed"]
if escalated:
    print("\nESCALATED (inspect gate_reports_<section>.json before Phase C):", escalated)
else:
    print("\nAll sections passed. Drafts + gate reports in:", PHASE_B_OUTPUT_DIR)

## B9 — Preview a generated section

In [ ]:
# ============================================================
# CELL B9 — PREVIEW A GENERATED SECTION
# ============================================================

PREVIEW_SECTION = RUN_SECTIONS[0]
draft = json.loads((PHASE_B_OUTPUT_DIR / f"draft_{PREVIEW_SECTION}.json").read_text(encoding="utf-8"))
for b in draft["blocks"]:
    prefix = "## " if b["type"] == "heading" else ""
    print(prefix + b["text"])
    if b["requirement_ids"]:
        print(f"    [satisfies: {', '.join(b['requirement_ids'][:6])}"
              + (" ..." if len(b["requirement_ids"]) > 6 else "") + "]")
    print()